# Assignment 3: Recurrent Neural Networks and LSTMs

**Deep Learning FS26**

---

## Part 1: Task Description

### Problem Description

In this assignment you will implement Recurrent Neural Networks (RNNs) and Long Short-Term Memory networks (LSTMs) using PyTorch. You will apply them to **sequence modeling** tasks: character-level language modeling and time-series forecasting. The goal is to understand how recurrent architectures handle temporal dependencies and the advantages of LSTMs over vanilla RNNs.

### Tasks Overview

1. **Vanilla RNN: Time-Series Forecasting**
   - Implement a single-layer RNN using `nn.RNN` to predict future values of a synthetic sine wave.
   - Use a sliding window approach to create training sequences.
   - Evaluate the model's ability to generalize to unseen time steps.

2. **LSTM: Time-Series Forecasting**
   - Replace the RNN cell with an LSTM (`nn.LSTM`) and compare performance.
   - Demonstrate that LSTM handles longer-range dependencies more effectively.

3. **Character-Level Language Model**
   - Train an LSTM on a small text corpus to learn character-level language patterns.
   - Use the trained model to **generate** new text sequences by sampling from the predicted character distribution.

### Possible Solutions

- The RNN and LSTM should both converge on the sine wave task, but LSTM typically achieves lower MSE for longer sequences.
- The generated text from the language model should start to resemble the style and vocabulary of the training corpus after sufficient training.
- You should observe the **vanishing gradient** problem with vanilla RNNs for long sequence lengths.

### Expected Plots

- **Time-series prediction plot**: The ground truth sine curve versus the model's prediction overlaid on the same axes.

  ```
  Value
   1.0 |  /\ Ground Truth
       | /  \
   0.0 |/    \____/  __Prediction
  -1.0 |              \/
        0  50 100 150 200  t
  ```

- **RNN vs LSTM loss comparison**: Training loss curves for both models on the same axes, showing that LSTM converges faster and to a lower loss.

- **Generated text sample**: A sequence of characters generated by the language model, visible in the notebook output.

- **Hidden state heatmap**: A heatmap of the hidden state activations over time steps for a single input sequence, revealing what the recurrent cell has "remembered".

## Part 2: Implementation

### Setup & Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

torch.manual_seed(42)
np.random.seed(42)

### Task 1: Vanilla RNN — Sine Wave Prediction

In [ ]:
# Generate a noisy sine wave
T = 1000
t = np.linspace(0, 8 * np.pi, T)
signal = np.sin(t) + 0.1 * np.random.randn(T)

# Normalize
signal = (signal - signal.mean()) / signal.std()

plt.figure(figsize=(12, 3))
plt.plot(t, signal, color='steelblue', linewidth=1)
plt.title('Input Signal (Noisy Sine Wave)')
plt.xlabel('Time')
plt.ylabel('Amplitude')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
def create_sequences(data, seq_length):
    """
    Create (input, target) pairs using a sliding window.
    Input: data[i : i+seq_length]
    Target: data[i+seq_length]
    """
    xs, ys = [], []
    for i in range(len(data) - seq_length):
        xs.append(data[i:i+seq_length])
        ys.append(data[i+seq_length])
    return np.array(xs), np.array(ys)

SEQ_LEN = 30
TRAIN_SIZE = 800

X, y = create_sequences(signal, SEQ_LEN)
X_train, y_train = X[:TRAIN_SIZE], y[:TRAIN_SIZE]
X_test, y_test   = X[TRAIN_SIZE:], y[TRAIN_SIZE:]

# Convert to PyTorch tensors: shape (batch, seq_len, input_size=1)
X_train_t = torch.FloatTensor(X_train).unsqueeze(-1).to(device)
y_train_t = torch.FloatTensor(y_train).unsqueeze(-1).to(device)
X_test_t  = torch.FloatTensor(X_test).unsqueeze(-1).to(device)
y_test_t  = torch.FloatTensor(y_test).unsqueeze(-1).to(device)

train_ds = TensorDataset(X_train_t, y_train_t)
train_dl = DataLoader(train_ds, batch_size=32, shuffle=True)

In [ ]:
class SimpleRNN(nn.Module):
    def __init__(self, input_size=1, hidden_size=64, num_layers=1):
        super(SimpleRNN, self).__init__()
        # TODO: Define RNN layer and output layer
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)
        self.fc  = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # TODO: Forward pass — use only the last hidden state
        out, _ = self.rnn(x)      # out: (batch, seq_len, hidden)
        out = self.fc(out[:, -1, :])  # Last time step
        return out


def train_sequence_model(model, train_dl, num_epochs=50):
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    losses = []
    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0
        for xb, yb in train_dl:
            optimizer.zero_grad()
            pred = model(xb)
            loss = criterion(pred, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # Gradient clipping
            optimizer.step()
            epoch_loss += loss.item()
        avg_loss = epoch_loss / len(train_dl)
        losses.append(avg_loss)
        if epoch % 10 == 0:
            print(f'Epoch {epoch:3d} | Loss: {avg_loss:.5f}')
    return losses


rnn_model = SimpleRNN(hidden_size=64).to(device)
print('Training Vanilla RNN...')
rnn_losses = train_sequence_model(rnn_model, train_dl, num_epochs=50)

### Task 2: LSTM — Sine Wave Prediction

In [ ]:
class SimpleLSTM(nn.Module):
    def __init__(self, input_size=1, hidden_size=64, num_layers=1):
        super(SimpleLSTM, self).__init__()
        # TODO: Replace RNN with LSTM
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc   = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # TODO: Forward pass — use only the last hidden state
        out, _ = self.lstm(x)
        out = self.fc(out[:, -1, :])
        return out


lstm_model = SimpleLSTM(hidden_size=64).to(device)
print('Training LSTM...')
lstm_losses = train_sequence_model(lstm_model, train_dl, num_epochs=50)

In [ ]:
# Compare RNN vs LSTM loss curves
plt.figure(figsize=(10, 4))
plt.plot(rnn_losses, label='Vanilla RNN', color='steelblue', linewidth=2)
plt.plot(lstm_losses, label='LSTM', color='coral', linewidth=2, linestyle='--')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Training Loss: Vanilla RNN vs LSTM')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Visualize predictions on test set
rnn_model.eval()
lstm_model.eval()

with torch.no_grad():
    rnn_preds  = rnn_model(X_test_t).cpu().numpy().flatten()
    lstm_preds = lstm_model(X_test_t).cpu().numpy().flatten()

ground_truth = y_test
time_idx = np.arange(len(ground_truth))

plt.figure(figsize=(12, 4))
plt.plot(time_idx, ground_truth, label='Ground Truth', color='black', linewidth=1.5)
plt.plot(time_idx, rnn_preds, label='RNN Prediction', color='steelblue', linestyle='--', linewidth=1)
plt.plot(time_idx, lstm_preds, label='LSTM Prediction', color='coral', linestyle=':', linewidth=1.5)
plt.xlabel('Time Step')
plt.ylabel('Amplitude')
plt.title('Sine Wave Prediction: RNN vs LSTM')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

rnn_mse  = np.mean((ground_truth - rnn_preds) ** 2)
lstm_mse = np.mean((ground_truth - lstm_preds) ** 2)
print(f'RNN  Test MSE: {rnn_mse:.5f}')
print(f'LSTM Test MSE: {lstm_mse:.5f}')

### Task 3: Character-Level Language Model

In [ ]:
# Small text corpus — a sample of English text
corpus = """
Deep learning is part of a broader family of machine learning methods based on artificial neural networks
with representation learning. Learning can be supervised, semi-supervised or unsupervised. Deep learning
architectures such as deep neural networks, recurrent neural networks, convolutional neural networks and
transformers have been applied to fields including computer vision, speech recognition, natural language
processing, machine translation, bioinformatics, drug design, and medical image analysis where they have
produced results comparable to and in some cases surpassing human expert performance.
""".strip().lower()

# Build character vocabulary
chars = sorted(set(corpus))
char_to_idx = {ch: idx for idx, ch in enumerate(chars)}
idx_to_char = {idx: ch for ch, idx in char_to_idx.items()}
vocab_size = len(chars)
print(f'Vocabulary size: {vocab_size}')
print(f'Vocab: {chars}')

In [ ]:
# Encode the corpus
CHAR_SEQ_LEN = 40
encoded = [char_to_idx[ch] for ch in corpus]

X_chars, y_chars = [], []
for i in range(len(encoded) - CHAR_SEQ_LEN):
    X_chars.append(encoded[i:i+CHAR_SEQ_LEN])
    y_chars.append(encoded[i+CHAR_SEQ_LEN])

X_chars = torch.LongTensor(X_chars).to(device)
y_chars = torch.LongTensor(y_chars).to(device)

char_ds = TensorDataset(X_chars, y_chars)
char_dl = DataLoader(char_ds, batch_size=64, shuffle=True)

In [ ]:
class CharLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_size, num_layers=2):
        super(CharLSTM, self).__init__()
        # TODO: Embedding layer + LSTM + output FC layer
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm      = nn.LSTM(embed_dim, hidden_size, num_layers, batch_first=True, dropout=0.3)
        self.fc        = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, hidden=None):
        # x: (batch, seq_len) of character indices
        x = self.embedding(x)         # (batch, seq_len, embed_dim)
        out, hidden = self.lstm(x, hidden)
        out = self.fc(out[:, -1, :])  # Predict next char from last time step
        return out, hidden


char_model = CharLSTM(vocab_size, embed_dim=32, hidden_size=128, num_layers=2).to(device)
char_optimizer = optim.Adam(char_model.parameters(), lr=1e-3)
char_criterion = nn.CrossEntropyLoss()

char_losses = []
num_char_epochs = 100

for epoch in range(1, num_char_epochs + 1):
    char_model.train()
    epoch_loss = 0
    for xb, yb in char_dl:
        char_optimizer.zero_grad()
        logits, _ = char_model(xb)
        loss = char_criterion(logits, yb)
        loss.backward()
        nn.utils.clip_grad_norm_(char_model.parameters(), max_norm=1.0)
        char_optimizer.step()
        epoch_loss += loss.item()
    avg_loss = epoch_loss / len(char_dl)
    char_losses.append(avg_loss)
    if epoch % 20 == 0:
        print(f'Epoch {epoch:4d} | Char Loss: {avg_loss:.4f}')

In [ ]:
# Text generation function
def generate_text(model, seed_text, length=200, temperature=1.0):
    """
    Generate text by sampling from the model's predicted character distribution.
    
    Args:
        model: trained CharLSTM
        seed_text: starting string (at least CHAR_SEQ_LEN characters)
        length: number of characters to generate
        temperature: sampling temperature (higher = more random)
    """
    model.eval()
    generated = seed_text
    current_seq = [char_to_idx.get(ch, 0) for ch in seed_text[-CHAR_SEQ_LEN:]]

    with torch.no_grad():
        for _ in range(length):
            x = torch.LongTensor([current_seq]).to(device)
            logits, _ = model(x)
            # Apply temperature and sample
            probs = torch.softmax(logits / temperature, dim=-1).squeeze()
            next_idx = torch.multinomial(probs, 1).item()
            generated += idx_to_char[next_idx]
            current_seq = current_seq[1:] + [next_idx]

    return generated


# Generate text with different temperatures
seed = corpus[:CHAR_SEQ_LEN]
print('=== Generated Text (temperature=0.5) ===')
print(generate_text(char_model, seed, length=200, temperature=0.5))
print()
print('=== Generated Text (temperature=1.0) ===')
print(generate_text(char_model, seed, length=200, temperature=1.0))

In [ ]:
# Plot character-level language model loss
plt.figure(figsize=(8, 4))
plt.plot(char_losses, color='mediumseagreen', linewidth=1.5)
plt.xlabel('Epoch')
plt.ylabel('Cross-Entropy Loss')
plt.title('Character-Level Language Model Training Loss')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Part 3: Experiments and Analysis

### Experiment 1: Effect of Sequence Length on RNN Performance

Train the vanilla RNN and LSTM on sequence lengths of `[10, 30, 60, 100]`. Report the test MSE for each sequence length and discuss how the models cope with longer dependencies.

In [ ]:
seq_lengths = [10, 30, 60, 100]
rnn_mse_results, lstm_mse_results = [], []

for seq_len in seq_lengths:
    # Prepare data
    X_s, y_s = create_sequences(signal, seq_len)
    split = 800
    X_tr = torch.FloatTensor(X_s[:split]).unsqueeze(-1).to(device)
    y_tr = torch.FloatTensor(y_s[:split]).unsqueeze(-1).to(device)
    X_te = torch.FloatTensor(X_s[split:]).unsqueeze(-1).to(device)
    y_te = torch.FloatTensor(y_s[split:]).unsqueeze(-1).to(device)

    ds = TensorDataset(X_tr, y_tr)
    dl = DataLoader(ds, batch_size=32, shuffle=True)

    # TODO: Train RNN and LSTM, record test MSE
    torch.manual_seed(42)
    rnn_s = SimpleRNN(hidden_size=64).to(device)
    train_sequence_model(rnn_s, dl, num_epochs=30)
    rnn_s.eval()
    with torch.no_grad():
        rnn_p = rnn_s(X_te).cpu().numpy().flatten()
    rnn_mse_results.append(np.mean((y_s[split:] - rnn_p) ** 2))

    torch.manual_seed(42)
    lstm_s = SimpleLSTM(hidden_size=64).to(device)
    train_sequence_model(lstm_s, dl, num_epochs=30)
    lstm_s.eval()
    with torch.no_grad():
        lstm_p = lstm_s(X_te).cpu().numpy().flatten()
    lstm_mse_results.append(np.mean((y_s[split:] - lstm_p) ** 2))

    print(f'SeqLen={seq_len:3d} | RNN MSE: {rnn_mse_results[-1]:.5f} | LSTM MSE: {lstm_mse_results[-1]:.5f}')

# Plot
plt.figure(figsize=(8, 4))
plt.plot(seq_lengths, rnn_mse_results, 'o-', label='Vanilla RNN', color='steelblue')
plt.plot(seq_lengths, lstm_mse_results, 's--', label='LSTM', color='coral')
plt.xlabel('Sequence Length')
plt.ylabel('Test MSE')
plt.title('Test MSE vs. Sequence Length')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# TODO: Interpret these results
# YOUR ANSWER HERE:

### Experiment 2: Hidden State Heatmap

Visualize the LSTM hidden state activations over time for a single input sequence. This reveals which time steps cause the strongest response in the recurrent cells.

In [ ]:
# Extract all hidden states (not just the last one)
class LSTMWithAllHidden(nn.Module):
    def __init__(self, input_size=1, hidden_size=64):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.fc   = nn.Linear(hidden_size, 1)

    def forward(self, x):
        all_hidden, _ = self.lstm(x)  # all_hidden: (batch, seq, hidden)
        return self.fc(all_hidden[:, -1, :]), all_hidden


# Train this model briefly
torch.manual_seed(42)
lstm_vis = LSTMWithAllHidden(hidden_size=32).to(device)
vis_optimizer = optim.Adam(lstm_vis.parameters(), lr=1e-3)
vis_criterion = nn.MSELoss()

for epoch in range(30):
    lstm_vis.train()
    for xb, yb in train_dl:
        vis_optimizer.zero_grad()
        pred, _ = lstm_vis(xb)
        loss = vis_criterion(pred, yb)
        loss.backward()
        vis_optimizer.step()

# Pick one test sequence
single_seq = X_test_t[:1]  # shape (1, seq_len, 1)
lstm_vis.eval()
with torch.no_grad():
    _, hidden_states = lstm_vis(single_seq)  # (1, seq_len, hidden)

hidden_np = hidden_states.squeeze(0).cpu().numpy()  # (seq_len, hidden_size)

plt.figure(figsize=(12, 5))
plt.imshow(hidden_np.T, aspect='auto', cmap='RdBu_r', interpolation='nearest')
plt.colorbar(label='Activation')
plt.xlabel('Time Step')
plt.ylabel('Hidden Unit')
plt.title('LSTM Hidden State Activations Over Time')
plt.tight_layout()
plt.show()

### Experiment 3: Temperature Sampling

Generate text at several temperature values (`0.2`, `0.5`, `1.0`, `1.5`) and observe how the diversity and coherence of the generated text changes.

In [ ]:
temperatures = [0.2, 0.5, 1.0, 1.5]
for temp in temperatures:
    print(f'\n=== Temperature = {temp} ===')
    print(generate_text(char_model, seed, length=150, temperature=temp))

# TODO: Describe the effect of temperature on the generated text
# - Low temperature (e.g. 0.2): ...
# - High temperature (e.g. 1.5): ...
# YOUR ANSWER HERE:

### Summary Questions

1. What is the vanishing gradient problem in RNNs? How does LSTM address it?
2. Explain the role of the forget gate, input gate, and output gate in an LSTM cell.
3. Why does a higher temperature in text generation lead to more diverse but less coherent output?
4. What are the main limitations of character-level language models compared to word-level or subword models?

**Your Answers:**

1. *TODO: Your answer here*

2. *TODO: Your answer here*

3. *TODO: Your answer here*

4. *TODO: Your answer here*